# Python Voice Assistant — OIBSIP Task 1
Run the setup cell first, then run the assistant cell. Allow microphone access when prompted.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y espeak ffmpeg
!pip -q install SpeechRecognition
print("Setup complete. Run the assistant cell next.")

## Voice assistant
Click **Start recording**, speak one command, then click the button again to stop recording. Try `Hello`, `What time is it?`, `What is today's date?`, `Search for Python tutorials`, `Remind me in 10 seconds to stretch`, `What is Python?`, or `Stop assistant`.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import speech_recognition as sr
import datetime
import webbrowser
import urllib.parse
import subprocess
import threading
import time
import re

CUSTOM_COMMANDS = {
    "what is python": "Python is a readable, general-purpose programming language.",
    "who created python": "Python was created by Guido van Rossum."
}

def speak(text):
    print("Assistant:", text)
    # Colab's espeak command produces audio in the runtime; text output is always shown.
    subprocess.run(["espeak", text], capture_output=True, check=False)

def record_audio():
    display(Javascript("""
    async function recordAudio() {
      const old = document.getElementById('voice-assistant-recorder');
      if (old) old.remove();
      const div = document.createElement('div');
      div.id = 'voice-assistant-recorder';
      const button = document.createElement('button');
      button.textContent = '🎤 Start recording';
      button.style.fontSize = '18px';
      button.style.padding = '12px';
      div.appendChild(button);
      document.body.appendChild(div);

      const stream = await navigator.mediaDevices.getUserMedia({audio:true});
      const recorder = new MediaRecorder(stream);
      let chunks = [];
      recorder.ondataavailable = e => chunks.push(e.data);
      recorder.start();
      button.textContent = '🔴 Recording — click to stop';
      await new Promise(resolve => button.onclick = resolve);
      recorder.stop();
      await new Promise(resolve => recorder.onstop = resolve);
      stream.getTracks().forEach(track => track.stop());
      div.remove();

      const blob = new Blob(chunks, {type: 'audio/webm'});
      const buffer = await blob.arrayBuffer();
      const bytes = new Uint8Array(buffer);
      let binary = '';
      bytes.forEach(b => binary += String.fromCharCode(b));
      return btoa(binary);
    }
    """))
    return b64decode(eval_js("recordAudio()"))

def listen():
    try:
        audio_bytes = record_audio()
        with open("recording.webm", "wb") as file:
            file.write(audio_bytes)
        conversion = subprocess.run(
            ["ffmpeg", "-y", "-i", "recording.webm", "-ar", "16000", "-ac", "1", "recording.wav"],
            capture_output=True
        )
        if conversion.returncode != 0:
            speak("I could not process that recording. Please try again.")
            return ""
        recognizer = sr.Recognizer()
        with sr.AudioFile("recording.wav") as source:
            audio = recognizer.record(source)
        command = recognizer.recognize_google(audio, language="en-US")
        print("You:", command)
        return command.lower()
    except sr.UnknownValueError:
        speak("Sorry, I did not understand. Please try again.")
    except sr.RequestError:
        speak("The speech recognition service is unavailable. Check your internet connection.")
    except Exception as error:
        print("Input error:", error)
        speak("Something went wrong. Please check microphone permission and try again.")
    return ""

def schedule_reminder(seconds, message):
    def notify():
        time.sleep(seconds)
        speak("Reminder: " + message)
    threading.Thread(target=notify, daemon=True).start()
    speak(f"Reminder set for {seconds} seconds.")

def handle_command(command):
    command = command.lower().strip()
    if not command:
        return True
    if any(phrase in command for phrase in ("stop assistant", "exit", "quit")):
        speak("Goodbye!")
        return False
    if any(word in command for word in ("hello", "hi", "hey")):
        speak("Hello! How can I help you?")
    elif "time" in command:
        speak("The current time is " + datetime.datetime.now().strftime("%I:%M %p"))
    elif "date" in command or "today" in command:
        speak("Today's date is " + datetime.datetime.now().strftime("%d %B %Y"))
    elif "remind me in" in command:
        match = re.search(r"remind me in\s+(\d+)\s+seconds?\s+(?:to\s+)?(.+)", command)
        if match:
            schedule_reminder(int(match.group(1)), match.group(2))
        else:
            speak("Please say, remind me in 10 seconds to take a break.")
    elif "search" in command or command.startswith("google "):
        query = command.replace("search for", "").replace("search", "").replace("google", "").strip()
        if query:
            webbrowser.open("https://www.google.com/search?q=" + urllib.parse.quote_plus(query))
            speak("I opened your web search.")
        else:
            speak("Please tell me what to search for.")
    elif command in CUSTOM_COMMANDS:
        speak(CUSTOM_COMMANDS[command])
    else:
        speak("I do not know that command yet. Try hello, time, date, search, or a reminder.")
    return True

def run_assistant():
    speak("Hello! Your Python voice assistant is ready.")
    while True:
        command = listen()
        if not handle_command(command):
            break

run_assistant()


## Privacy note
Speech is transcribed using the online Google speech-recognition service through SpeechRecognition. Avoid speaking sensitive information. The microphone is accessed only after you start recording and grant browser permission.